# Aviation Accidents: Data Cleaning and Feature Engineering

**Analyst:** Atilio Barreda

## Business problem

An airline/aircraft insurer wants evidence-based recommendations for professionally built airplanes that may still be active. The analysis therefore:

- restricts the data to **airplanes**
- excludes **amateur-built aircraft**
- keeps accidents from **1983 onward**
- estimates the fraction of people aboard who were fatally or seriously injured
- records whether an aircraft was destroyed
- retains only manufacturers with enough observations to support comparison
- creates a unique make/model identifier for later recommendations

The dataset records accidents, not total flights or flight hours. Results therefore describe outcomes **conditional on a recorded accident**; they are not exposure-adjusted accident probabilities.

### Make relevant library imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## Data Loading and Inspection

### Load the raw dataset and inspect its structure

The repository normally stores the source file at `data/AviationData.csv`. The path resolution below also supports the CSV being placed beside the notebook.

In [ ]:
raw_candidates = [
    Path("data/AviationData.csv"),
    Path("AviationData.csv"),
    Path("../data/AviationData.csv"),
]

raw_data_path = next((path for path in raw_candidates if path.exists()), None)

if raw_data_path is None:
    raise FileNotFoundError(
        "AviationData.csv was not found. Place it in data/AviationData.csv "
        "or in the same folder as this notebook."
    )

df_raw = pd.read_csv(raw_data_path, encoding="latin-1", low_memory=False)
df = df_raw.copy()

print(f"Loaded: {raw_data_path.resolve()}")
print(f"Raw shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
display(df.head())

inspection = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
}).sort_values("missing_pct", ascending=False)

display(inspection)
display(df.describe(include="all").T)

## Data Cleaning

### Filter aircraft and events to match the client brief

The client requested professionally built airplanes that could plausibly remain active.

Cleaning decisions:

1. Normalize `Aircraft.Category` and retain only `"AIRPLANE"`.
2. Normalize `Amateur.Built` and retain only `"NO"`.
3. Parse `Event.Date` as a date and retain events on or after **January 1, 1983**.
4. Keep the original row-level accident records; no accident is duplicated during filtering.

In [ ]:
def first_existing_column(frame, candidates):
    for candidate in candidates:
        if candidate in frame.columns:
            return candidate
    raise KeyError(f"None of the expected columns were found: {candidates}")

category_col = first_existing_column(df, ["Aircraft.Category", "Aircraft Category"])
amateur_col = first_existing_column(df, ["Amateur.Built", "Amateur Built"])
date_col = first_existing_column(df, ["Event.Date", "Event Date"])

df[category_col] = (
    df[category_col]
    .astype("string")
    .str.strip()
    .str.upper()
)

df[amateur_col] = (
    df[amateur_col]
    .astype("string")
    .str.strip()
    .str.upper()
)

df["Event.Date"] = pd.to_datetime(df[date_col], errors="coerce")

filter_mask = (
    df[category_col].eq("AIRPLANE")
    & df[amateur_col].eq("NO")
    & df["Event.Date"].ge(pd.Timestamp("1983-01-01"))
)

df = df.loc[filter_mask].copy()

print(f"Shape after client filters: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print(f"Date range: {df['Event.Date'].min().date()} through {df['Event.Date'].max().date()}")

### Clean and construct the key safety measures

The two primary outcomes are:

- **Fatal/serious injury fraction:** fatal plus serious injuries divided by the estimated number of people aboard.
- **Destroyed:** a binary indicator equal to 1 when the aircraft damage category is `"Destroyed"`.

A row with no injury counts at all is treated as unknown rather than as zero. A fraction is calculated only when the estimated number aboard is greater than zero.

**Construct a metric for fatal and serious injuries**

In [ ]:
injury_source_columns = {
    "Fatal.Injuries": ["Total.Fatal.Injuries", "Fatal.Injuries"],
    "Serious.Injuries": ["Total.Serious.Injuries", "Serious.Injuries"],
    "Minor.Injuries": ["Total.Minor.Injuries", "Minor.Injuries"],
    "Uninjured": ["Total.Uninjured", "Uninjured"],
}

source_names = {}
for canonical_name, candidates in injury_source_columns.items():
    source_col = first_existing_column(df, candidates)
    source_names[canonical_name] = source_col
    df[canonical_name] = pd.to_numeric(df[source_col], errors="coerce")

reported_injury_fields = df[list(source_names.values())].notna().sum(axis=1)

df["People.Aboard"] = (
    df[["Fatal.Injuries", "Serious.Injuries", "Minor.Injuries", "Uninjured"]]
    .fillna(0)
    .sum(axis=1)
)
df.loc[reported_injury_fields.eq(0), "People.Aboard"] = np.nan

df["Fatal.Serious.Injuries"] = (
    df[["Fatal.Injuries", "Serious.Injuries"]]
    .fillna(0)
    .sum(axis=1)
)
df.loc[reported_injury_fields.eq(0), "Fatal.Serious.Injuries"] = np.nan

df["Fatal.Serious.Injury.Fraction"] = (
    df["Fatal.Serious.Injuries"]
    .div(df["People.Aboard"].where(df["People.Aboard"].gt(0)))
    .clip(lower=0, upper=1)
)

display(
    df[
        [
            "Fatal.Injuries",
            "Serious.Injuries",
            "Minor.Injuries",
            "Uninjured",
            "People.Aboard",
            "Fatal.Serious.Injuries",
            "Fatal.Serious.Injury.Fraction",
        ]
    ].describe()
)

**Aircraft damage**

Damage labels are stripped and normalized. Missing or placeholder labels become `"Unknown"`. `Destroyed` is stored as a 0/1 integer so its group mean is the destruction rate.

In [ ]:
damage_col = first_existing_column(
    df,
    ["Aircraft.damage", "Aircraft.Damage", "Aircraft Damage"],
)

damage_clean = (
    df[damage_col]
    .astype("string")
    .str.strip()
    .str.title()
    .replace({
        "": pd.NA,
        "Unk": pd.NA,
        "Unknown": pd.NA,
    })
    .fillna("Unknown")
)

df["Aircraft.Damage"] = damage_clean
df["Destroyed"] = df["Aircraft.Damage"].eq("Destroyed").astype("int8")

display(df["Aircraft.Damage"].value_counts(dropna=False).to_frame("accidents"))
print(f"Overall destruction rate in the filtered sample: {df['Destroyed'].mean():.3%}")

### Investigate and clean `Make`

Manufacturer names contain inconsistent capitalization, spacing, and corporate suffixes. The code:

- converts names to uppercase
- collapses repeated whitespace
- standardizes several high-frequency manufacturer aliases
- removes missing/placeholder makes
- retains makes represented by **at least 50 accidents**, as required for robust make-level comparisons

In [ ]:
df["Make"] = (
    df["Make"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

make_patterns = [
    (r"\bCESSNA\b", "CESSNA"),
    (r"\bPIPER\b", "PIPER"),
    (r"\bBEECH(?:CRAFT)?\b", "BEECH"),
    (r"\bBOEING\b", "BOEING"),
    (r"\bAIRBUS\b", "AIRBUS"),
    (r"\bMCDONNELL[ -]?DOUGLAS\b", "MCDONNELL DOUGLAS"),
    (r"\bDOUGLAS\b", "DOUGLAS"),
    (r"\bMOONEY\b", "MOONEY"),
    (r"\bCIRRUS\b", "CIRRUS"),
    (r"\bGRUMMAN\b", "GRUMMAN"),
    (r"\bEMBRAER\b", "EMBRAER"),
    (r"\bBOMBARDIER\b", "BOMBARDIER"),
]

for pattern, standardized_name in make_patterns:
    match_mask = df["Make"].str.contains(pattern, regex=True, na=False)
    df.loc[match_mask, "Make"] = standardized_name

df["Make"] = df["Make"].replace(
    {"": pd.NA, "UNKNOWN": pd.NA, "UNK": pd.NA, "NONE": pd.NA}
)
df = df.dropna(subset=["Make"]).copy()

make_counts_before_threshold = df["Make"].value_counts()
valid_makes = make_counts_before_threshold[
    make_counts_before_threshold >= 50
].index

df = df[df["Make"].isin(valid_makes)].copy()

print(f"Makes retained: {df['Make'].nunique():,}")
display(df["Make"].value_counts().to_frame("accidents").head(25))

### Inspect and clean `Model`

Model labels are normalized to uppercase and repeated whitespace is collapsed. Rows without a usable model are removed. Because model labels can repeat across manufacturers, `Plane.Type` combines `Make` and `Model` into a unique make/model identifier.

In [ ]:
df["Model"] = (
    df["Model"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .replace({"": pd.NA, "UNKNOWN": pd.NA, "UNK": pd.NA, "NONE": pd.NA})
)

df = df.dropna(subset=["Model"]).copy()
df["Plane.Type"] = df["Make"] + " " + df["Model"]

print(f"Unique plane types: {df['Plane.Type'].nunique():,}")
display(df["Plane.Type"].value_counts().to_frame("accidents").head(25))

### Clean other potentially explanatory columns

- String categories are stripped, uppercased, and given a consistent `"UNKNOWN"` label when missing.
- `Number.of.Engines` is converted to numeric.
- Zero, negative, and implausibly high engine counts are treated as missing rather than as real values.
- Missing values are retained because dropping every incomplete row would remove substantial information and bias the remaining sample.

In [ ]:
categorical_columns = [
    "Engine.Type",
    "Weather.Condition",
    "Purpose.of.flight",
    "Broad.phase.of.flight",
]

for column in categorical_columns:
    if column in df.columns:
        df[column] = (
            df[column]
            .astype("string")
            .str.strip()
            .str.upper()
            .replace({
                "": pd.NA,
                "UNK": pd.NA,
                "UNKNOWN": pd.NA,
                "NONE": pd.NA,
                "N/A": pd.NA,
            })
            .fillna("UNKNOWN")
        )

if "Number.of.Engines" in df.columns:
    df["Number.of.Engines"] = pd.to_numeric(
        df["Number.of.Engines"], errors="coerce"
    )
    invalid_engine_count = (
        df["Number.of.Engines"].le(0)
        | df["Number.of.Engines"].gt(8)
    )
    df.loc[invalid_engine_count, "Number.of.Engines"] = np.nan

for column in categorical_columns + ["Number.of.Engines"]:
    if column in df.columns:
        print(f"\n{column}")
        display(df[column].value_counts(dropna=False).head(20).to_frame("accidents"))

### Remove columns with excessive missingness

Per the assignment, original columns are retained when they have **more than 20,000 non-null values**. Essential identifiers, outcomes, and factor columns created or required for analysis are preserved even when a derived field has fewer than 20,000 valid observations.

In [ ]:
non_null_counts = df.notna().sum()
threshold_columns = non_null_counts[non_null_counts > 20_000].index.tolist()

required_columns = [
    "Event.Date",
    "Make",
    "Model",
    "Plane.Type",
    "Aircraft.Damage",
    "Destroyed",
    "Fatal.Injuries",
    "Serious.Injuries",
    "Minor.Injuries",
    "Uninjured",
    "People.Aboard",
    "Fatal.Serious.Injuries",
    "Fatal.Serious.Injury.Fraction",
    "Engine.Type",
    "Weather.Condition",
    "Number.of.Engines",
    "Purpose.of.flight",
    "Broad.phase.of.flight",
]

keep_columns = []
for column in list(df.columns) + required_columns:
    if (
        column in df.columns
        and column not in keep_columns
        and (column in threshold_columns or column in required_columns)
    ):
        keep_columns.append(column)

removed_columns = [column for column in df.columns if column not in keep_columns]
df_clean = df[keep_columns].copy()

print(f"Removed {len(removed_columns)} sparse columns:")
print(removed_columns)
print(f"Final cleaned shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]:,} columns")

final_quality = pd.DataFrame({
    "dtype": df_clean.dtypes.astype(str),
    "non_null": df_clean.notna().sum(),
    "missing_pct": (df_clean.isna().mean() * 100).round(2),
})
display(final_quality)

### Save the cleaned dataframe

The cleaned CSV is written to the repository's `data` folder when available. The analysis notebook searches both `data/AviationData_cleaned.csv` and the notebook directory.

In [ ]:
output_directory = (
    raw_data_path.parent
    if raw_data_path.parent.name == "data"
    else Path("data")
)

output_directory.mkdir(parents=True, exist_ok=True)
cleaned_data_path = output_directory / "AviationData_cleaned.csv"

df_clean.to_csv(cleaned_data_path, index=False)

print(f"Saved cleaned data to: {cleaned_data_path.resolve()}")
print(f"Saved shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]:,} columns")